##### 选择引擎

In [ ]:
class MultiFactorStockSelector:
    """财务+波浪+行业轮动三因子选股引擎"""
    
    def __init__(self, financial_scorer: IndustryAdjustedFinancialScorer,
                 wave_scorer_factory,  # 波浪评分器工厂函数
                 top_n: int = 30,
                 industry_diversity: bool = True):
        self.financial_scorer = financial_scorer
        self.wave_scorer_factory = wave_scorer_factory
        self.top_n = top_n
        self.industry_diversity = industry_diversity  # 是否强制行业分散
    
    def select_top_stocks(self, 
                         market_data: Dict[str, pd.DataFrame],  # {stock_code: df}
                         financial_data: pd.DataFrame,
                         industry_index_data: Dict[str, pd.DataFrame] = None) -> pd.DataFrame:
        """
        全市场选股主流程
        :return: Top N股票列表（含综合得分、财务得分、波浪得分、行业等）
        """
        # 1. 股票池过滤（流动性+基本面）
        universe = self._filter_universe(market_data, financial_data)
        
        # 2. 并行计算财务评分（向量化）
        financial_scores = self.financial_scorer.calculate_financial_score(financial_data)
        
        # 3. 并行计算波浪信号（多进程加速）
        wave_signals = self._parallel_wave_scoring(universe, market_data)
        
        # 4. 行业景气度计算（可选）
        industry_momentum = self._calculate_industry_momentum(industry_index_data) if industry_index_data else {}
        
        # 5. 综合评分
        results = []
        for stock_code in universe:
            fin_score = financial_scores.loc[financial_scores['stock_code']==stock_code, 'financial_score'].values[0] if not financial_scores.empty else 50
            wave_sig = wave_signals.get(stock_code, {'wave_score': 0})
            ind_mom = industry_momentum.get(universe[stock_code]['industry3'], 50)
            
            # 综合得分 = 财务×0.4 + 波浪×0.5 + 行业×0.1
            composite_score = fin_score * 0.4 + wave_sig['wave_score'] * 0.5 + ind_mom * 0.1
            
            results.append({
                'stock_code': stock_code,
                'stock_name': universe[stock_code]['name'],
                'industry3': universe[stock_code]['industry3'],
                'financial_score': round(fin_score, 2),
                'wave_score': round(wave_sig['wave_score'], 2),
                'wave_phase': wave_sig.get('current_phase', 'N/A'),
                'industry_momentum': round(ind_mom, 2),
                'composite_score': round(composite_score, 2),
                'confidence': wave_sig.get('confidence', 0),
                'target_price': wave_sig.get('target_price', None)
            })
        
        # 6. 排序与行业分散约束
        results_df = pd.DataFrame(results).sort_values('composite_score', ascending=False)
        
        if self.industry_diversity:
            results_df = self._apply_industry_diversity_constraint(results_df)
        
        return results_df.head(self.top_n).reset_index(drop=True)
    
    def _apply_industry_diversity_constraint(self, df: pd.DataFrame) -> pd.DataFrame:
        """行业分散约束：单三级行业不超过Top N的20%"""
        max_per_industry = max(1, int(self.top_n * 0.2))
        selected = []
        industry_count = {}
        
        for _, row in df.iterrows():
            ind = row['industry3']
            if industry_count.get(ind, 0) < max_per_industry:
                selected.append(row)
                industry_count[ind] = industry_count.get(ind, 0) + 1
            if len(selected) >= self.top_n:
                break
        
        return pd.DataFrame(selected) if selected else df.head(self.top_n)
    
    def _filter_universe(self, market_data: Dict, financial_data: pd.DataFrame) -> Dict:
        """股票池过滤：流动性+基本面+交易状态"""
        universe = {}
        for code, df in market_data.items():
            # 流动性过滤：20日均成交额 > 5000万
            if df['vol'].tail(20).mean() * df['close'].tail(20).mean() < 5e7:
                continue
            
            # 基本面过滤：最近4季度财报完整
            if code not in financial_data['stock_code'].values:
                continue
            
            # 交易状态：非ST、非停牌
            if 'ST' in df['name'].iloc[-1] or df['close'].isna().any():
                continue
            
            # 行业映射
            industry2 = financial_data.loc[financial_data['stock_code']==code, 'industry'].values[0]
            industry3 = self.financial_scorer.industry_hierarchy.get(industry2, ('Unknown', 'Unknown'))[1]
            
            universe[code] = {
                'name': df['name'].iloc[-1],
                'industry3': industry3,
                'df': df
            }
        
        return universe